In [1]:
import re
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("Maxscha/commitbench")

# Enhanced cleaning function for the 'message' column
def clean_message(message, seen_messages):
    # Remove patterns like #<I>, <I>, HTML-like tags
    message = re.sub(r'#<I>\s*<I>', '', message)
    message = re.sub(r'<[^>]+>', '', message)

    # Remove specific patterns for issues and tasks
    message = re.sub(r'\(#.*?\)', '', message)  # Remove issue references in parentheses
    message = re.sub(r'closes #\d+', '', message)  # Remove "closes #" patterns
    message = re.sub(r'task\s*#:\s*', '', message, flags=re.IGNORECASE)  # Remove task references

    # Remove redundant punctuation and extra symbols
    message = re.sub(r'\*+', '', message)  # Remove excessive asterisks
    message = re.sub(r'-+', ' ', message)  # Replace multiple dashes with a space

    # Normalize spaces around punctuation
    message = re.sub(r'\s*\.\s*', '. ', message)  # Ensure space before/after periods
    message = re.sub(r'\s+', ' ', message).strip()  # Remove extra spaces

    # Normalize specific terms
    message = re.sub(r'\bnon empty\b', 'non-empty', message)  # Standardize "non empty" to "non-empty"
    message = re.sub(r'\bsetup\. toml\b', 'setup.toml', message)  # Standardize "setup. toml" to "setup.toml"
    message = re.sub(r'\bsetup\. py\b', 'setup.py', message)  # Standardize "setup. py" to "setup.py"

    # Fully spell out abbreviated terms (example)
    message = re.sub(r'\bcomput\.\.\.\b', 'computation of constructors', message)  # Example of expanding an abbreviation

    # Check for incomplete phrases and replace or clarify if necessary
    if message == "bumpy to":
        message = "description unclear, please clarify"  # Replace with a clarifying statement

    # Check for duplicates and consolidate messages
    if message in seen_messages:
        return None  # Return None for duplicate messages
    seen_messages.add(message)  # Add the message to the seen set

    # Lowercase the message
    return message.lower()

# Set to track seen messages
seen_messages = set()

# Apply the cleaning function to the 'message' column for all splits
cleaned_dataset = dataset.map(lambda x: {"message": clean_message(x["message"], seen_messages)}, batched=False)

# Filter out None values (duplicates or invalid messages)
cleaned_dataset = cleaned_dataset.filter(lambda x: x["message"] is not None)

In [2]:
from transformers import BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments

# Load pre-trained model and tokenizer
model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

MAX_DIFF_LENGTH = 512  # For the 'diff' input
MAX_MESSAGE_LENGTH = 128  # For the 'message' output

# Tokenize the dataset (with separate max lengths for diff and message)
def tokenize_function(examples):
    # Tokenize the diffs as input with a max length of 512
    model_inputs = tokenizer(
        examples["diff"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_DIFF_LENGTH  # Max length for diff
    )
    
    # Tokenize the commit messages as target/output labels with a max length of 128
    labels = tokenizer(
        text_target=examples["message"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_MESSAGE_LENGTH  # Max length for message
    )
    
    # Assign tokenized commit messages to the 'labels' key
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the tokenization to the cleaned dataset
tokenized_datasets = cleaned_dataset.map(tokenize_function, batched=True)

In [3]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# Define training arguments
training_args = TrainingArguments(
    output_dir="./teamspace/uploads/result",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=70,
    per_device_eval_batch_size=70,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir='./teamspace/uploads/logs',
    logging_steps=100,
    save_total_limit=1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    run_name="swiftcommit-finetuned",
    fp16=True,
)

In [ ]:
# Set up the trainer with early stopping callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Adjust patience as needed
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.467400,0.440747
2,0.441200,0.420822
3,0.422000,0.410392
4,0.418500,0.405143
5,0.396800,0.400115
6,0.406600,0.397617
7,0.392400,0.395442
8,0.388500,0.393661


/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


In [5]:
print(cleaned_dataset['train']['message'][:10])  

['refactoring to use phasediagram. pd_coords', 'refactor(replset): reduce server count when destroyed', 'if there is no repository data, display a nicer message with details on how to proceed to potentially fix it', 'bibformat: hdref processing bug fix fixes bug that would occur with empty records when processing hdref. (closes #)', 'block titles fast fix implementation of isempty method, to check, does property have non-empty raw value or not (raw means value without cms editable div wrappers)', 'add additional cbor test for usetagformaps', 'decorate ecsrunlauncher as experimental summary: now that', "[build] add tests' requires in setup.py", 'map lazy/upward dylib to proper load command both `lc_lazy_load_dylib` and `lc_load_upward_dylib` are represented as a `dylibcommand` internally as can be verified by creating test dylibs with the ` lazy_library` and ` upward_library` linker arguments.', 'consume bucket once when blocking property is enabled.']
